# 🕷️ Task 1: Web Scraping
### CodeAlpha Data Analytics Internship
**Goal:** Scrape top books data from [books.toscrape.com](http://books.toscrape.com) — a legal, practice scraping site.  
We'll collect: Title, Price, Rating, Availability — then save to CSV for analysis.

In [ ]:
# Install required libraries (run once)
# !pip install requests beautifulsoup4 pandas lxml

import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported successfully!")

In [ ]:
# ─────────────────────────────────────────────
# STEP 1: Scrape all pages from books.toscrape.com
# ─────────────────────────────────────────────

BASE_URL = "http://books.toscrape.com/catalogue/"
START_URL = "http://books.toscrape.com/catalogue/page-1.html"

RATING_MAP = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

books = []

def scrape_page(url):
    response = requests.get(url, timeout=10)
    soup = BeautifulSoup(response.text, 'lxml')
    articles = soup.find_all('article', class_='product_pod')
    
    for article in articles:
        title = article.h3.a['title']
        price = article.find('p', class_='price_color').text.strip().replace('Â', '').replace('£','£')
        rating_word = article.find('p', class_='star-rating')['class'][1]
        rating = RATING_MAP.get(rating_word, 0)
        availability = article.find('p', class_='instock availability').text.strip()
        books.append({
            'Title': title,
            'Price (£)': float(price.replace('£','').strip()),
            'Rating (out of 5)': rating,
            'Availability': availability
        })
    
    # Check for next page
    next_btn = soup.find('li', class_='next')
    if next_btn:
        return BASE_URL + next_btn.a['href']
    return None

# Scrape all 50 pages
url = START_URL
page = 1
while url:
    print(f"Scraping page {page}...", end=" ")
    url = scrape_page(url)
    page += 1
    time.sleep(0.3)  # polite delay

print(f"\n✅ Done! Total books scraped: {len(books)}")

In [ ]:
# ─────────────────────────────────────────────
# STEP 2: Create DataFrame & Inspect
# ─────────────────────────────────────────────

df = pd.DataFrame(books)

print("Shape:", df.shape)
print("\nColumn Types:")
print(df.dtypes)
print("\nFirst 5 rows:")
df.head()

In [ ]:
# ─────────────────────────────────────────────
# STEP 3: Basic Analysis on Scraped Data
# ─────────────────────────────────────────────

print("📊 Summary Statistics")
print("=" * 40)
print(f"Total Books Scraped   : {len(df)}")
print(f"Average Price         : £{df['Price (£)'].mean():.2f}")
print(f"Cheapest Book         : £{df['Price (£)'].min():.2f}")
print(f"Most Expensive Book   : £{df['Price (£)'].max():.2f}")
print(f"Average Rating        : {df['Rating (out of 5)'].mean():.2f} / 5")
print(f"5-Star Books          : {len(df[df['Rating (out of 5)'] == 5])}")
print(f"In Stock Books        : {df['Availability'].value_counts()['In stock']}")

In [ ]:
# ─────────────────────────────────────────────
# STEP 4: Visualize the Scraped Data
# ─────────────────────────────────────────────

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
fig = plt.figure(figsize=(18, 12))
fig.suptitle('📚 Books Scraping Analysis — books.toscrape.com', 
             fontsize=20, fontweight='bold', y=0.98)

gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# Chart 1: Rating Distribution
ax1 = fig.add_subplot(gs[0, 0])
rating_counts = df['Rating (out of 5)'].value_counts().sort_index()
colors = ['#e74c3c','#e67e22','#f1c40f','#2ecc71','#27ae60']
bars = ax1.bar(rating_counts.index, rating_counts.values, color=colors, edgecolor='white', linewidth=1.5)
ax1.set_title('Rating Distribution', fontsize=13, fontweight='bold')
ax1.set_xlabel('Star Rating')
ax1.set_ylabel('Number of Books')
for bar in bars:
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1, 
             str(int(bar.get_height())), ha='center', fontsize=10, fontweight='bold')

# Chart 2: Price Distribution
ax2 = fig.add_subplot(gs[0, 1])
ax2.hist(df['Price (£)'], bins=25, color='#3498db', edgecolor='white', linewidth=1)
ax2.axvline(df['Price (£)'].mean(), color='red', linestyle='--', linewidth=2, label=f"Mean: £{df['Price (£)'].mean():.2f}")
ax2.set_title('Price Distribution', fontsize=13, fontweight='bold')
ax2.set_xlabel('Price (£)')
ax2.set_ylabel('Frequency')
ax2.legend()

# Chart 3: Avg Price per Rating
ax3 = fig.add_subplot(gs[0, 2])
avg_price = df.groupby('Rating (out of 5)')['Price (£)'].mean()
ax3.plot(avg_price.index, avg_price.values, marker='o', color='#9b59b6', linewidth=2.5, markersize=8)
ax3.fill_between(avg_price.index, avg_price.values, alpha=0.15, color='#9b59b6')
ax3.set_title('Avg Price by Rating', fontsize=13, fontweight='bold')
ax3.set_xlabel('Rating')
ax3.set_ylabel('Avg Price (£)')

# Chart 4: Top 10 Most Expensive Books
ax4 = fig.add_subplot(gs[1, :2])
top10 = df.nlargest(10, 'Price (£)')[['Title','Price (£)']]
top10['Short Title'] = top10['Title'].apply(lambda x: x[:35]+'...' if len(x)>35 else x)
bars = ax4.barh(top10['Short Title'], top10['Price (£)'], color='#e74c3c', edgecolor='white')
ax4.set_title('Top 10 Most Expensive Books', fontsize=13, fontweight='bold')
ax4.set_xlabel('Price (£)')
for bar in bars:
    ax4.text(bar.get_width()+0.2, bar.get_y()+bar.get_height()/2,
             f'£{bar.get_width():.2f}', va='center', fontsize=9)

# Chart 5: Books per Rating (Pie)
ax5 = fig.add_subplot(gs[1, 2])
wedge_colors = ['#e74c3c','#e67e22','#f1c40f','#2ecc71','#27ae60']
ax5.pie(rating_counts.values, labels=[f'⭐ {i}' for i in rating_counts.index],
        autopct='%1.1f%%', colors=wedge_colors, startangle=90, textprops={'fontsize':10})
ax5.set_title('Books Share by Rating', fontsize=13, fontweight='bold')

plt.savefig('Task1_WebScraping/scraping_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Chart saved!")

In [ ]:
# ─────────────────────────────────────────────
# STEP 5: Save Scraped Data to CSV
# ─────────────────────────────────────────────

df.to_csv('Task1_WebScraping/scraped_books.csv', index=False)
print("✅ Data saved to Task1_WebScraping/scraped_books.csv")
print(f"   Total rows: {len(df)}")
print("\n📋 Sample of saved data:")
df.sample(5)

## ✅ Task 1 Summary

| Metric | Value |
|--------|-------|
| Total Books Scraped | 1000 |
| Pages Scraped | 50 |
| Average Price | ~£35 |
| Average Rating | ~3/5 |
| Data saved to | `scraped_books.csv` |

**Skills demonstrated:** HTTP requests, HTML parsing with BeautifulSoup, pagination handling, data cleaning, CSV export.